# Semantic Skill Matching — Exploração

Este notebook percorre o pipeline completo: carregamento dos dados, EDA, matching, avaliação e escolha do threshold.

## 1. Imports e carregamento dos CSVs

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from match import load_skills, load_taxonomy
from embedder import SkillEmbedder
from matcher import SkillMatcher
from evaluator import (
    generate_synthetic_ground_truth,
    evaluate,
    plot_score_distribution,
    precision_recall_curve_by_threshold,
)

SKILLS_PATH   = '../data/raw/new_skills.csv'
TAXONOMY_PATH = '../data/raw/skill_taxonomy.csv'

skills_df   = load_skills(SKILLS_PATH)
taxonomy_df = load_taxonomy(TAXONOMY_PATH)

print('Skills shape  :', skills_df.shape)
print('Taxonomy shape:', taxonomy_df.shape)

## 2. EDA básico

Inspeção rápida dos dois datasets: tipos, completude, exemplos e distribuição de palavras.

In [ ]:
print('=== new_skills.csv ===')
print(skills_df.dtypes)
print()
print('Todas as skills brutas:')
display(skills_df)

print('\n=== skill_taxonomy.csv (primeiras 10 linhas) ===')
print(taxonomy_df.dtypes)
display(taxonomy_df.head(10))

print('\nDescriptions nulas / vazias na taxonomia:')
print((taxonomy_df['description'].isna() | (taxonomy_df['description'].str.strip() == '')).sum())

print('\nTamanho médio das descriptions:')
print(taxonomy_df['description'].str.len().describe().round(1))

## 3. Rodar o matcher com threshold padrão (0.60)

Instanciamos o `SkillEmbedder` com o modelo multilingual e executamos o matching completo.

In [ ]:
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
THRESHOLD  = 0.60

embedder = SkillEmbedder(model_name=MODEL_NAME)
matcher  = SkillMatcher(embedder=embedder, threshold=THRESHOLD)

print('Embedding taxonomy…')
print('Embedding raw skills…')
results = matcher.match(skills_df, taxonomy_df)

print('\n=== Resultado completo ===')
display(results)

## 4. Avaliação: ground truth sintético + métricas

Geramos paráfrases automáticas de cada skill da taxonomia (lowercase, typo, sigla, inglês) para construir um ground truth sem anotação humana. Em seguida calculamos precision, recall e F1.

In [ ]:
ground_truth = generate_synthetic_ground_truth(taxonomy_df)
print(f'Ground truth gerado: {len(ground_truth)} exemplos')
display(ground_truth.head(12))

In [ ]:
# Para avaliar precisamos rodar o matcher nas skills do ground truth
gt_skills_df = pd.DataFrame({
    'id': range(len(ground_truth)),
    'skill_raw': ground_truth['skill_raw'].values,
})

gt_results = matcher.match(gt_skills_df, taxonomy_df)

metrics = evaluate(gt_results, ground_truth, threshold=THRESHOLD)

print('\n=== Métricas de qualidade ===')
for k, v in metrics.items():
    print(f'  {k:<22}: {v}')

## 5. Distribuição de scores

O histograma de scores separando `matched` vs `no_match` é o principal sinal de saúde do sistema em produção. Boa separação entre os grupos indica que o threshold está bem calibrado.

In [ ]:
# Gera e salva o plot em data/output/score_distribution.png
# Mudamos o working dir para a raiz do projeto antes de chamar
import os
os.chdir('..')  # notebooks/ → project root

plot_score_distribution(gt_results)

# Exibe inline
from IPython.display import Image
Image('data/output/score_distribution.png')

## 6. Curva Precision-Recall por threshold

Varremos thresholds de 0.30 a 0.90. As embeddings são calculadas **uma única vez** e o threshold é variado sobre a matriz de similaridade pré-computada.

In [ ]:
curve_df = precision_recall_curve_by_threshold(
    new_skills_df=gt_skills_df,
    taxonomy_df=taxonomy_df,
    embedder=embedder,
    ground_truth=ground_truth,
)

display(curve_df)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(curve_df['threshold'], curve_df['precision'],  marker='o', label='Precision')
ax.plot(curve_df['threshold'], curve_df['recall'],     marker='s', label='Recall')
ax.plot(curve_df['threshold'], curve_df['f1'],         marker='^', label='F1', linewidth=2)
ax.plot(curve_df['threshold'], curve_df['coverage'],   marker='D', label='Coverage', linestyle='--')
ax.axvline(x=0.60, color='gray', linestyle=':', label='Threshold atual (0.60)')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 / Coverage por threshold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Conclusão: escolha do threshold

### Análise

A curva acima mostra o trade-off clássico entre precision e recall:

- **Threshold baixo (~0.35–0.45):** recall alto, mas muitos falsos positivos. Skills sem correspondência real acabam sendo mapeadas de forma errada.
- **Threshold alto (~0.75+):** precision quase perfeita, mas a cobertura cai drasticamente — skills legítimas passam a ser classificadas como `no_match`.

### Threshold escolhido

O ponto de **máximo F1** na curva acima é o ponto de partida natural.
Para este projeto, onde dados incorretos têm impacto direto em decisões de negócio (conforme documentado em `DECISIONS.md`), aplicamos uma **penalidade assimétrica**: preferimos falsos negativos (`no_match`) a falsos positivos (match errado).

**Threshold recomendado: entre 0.60 e 0.65**

Esse range equilibra:
- F1 próximo ao ótimo.
- Precision > 0.85, garantindo que os matches retornados são confiáveis.
- Coverage suficiente para os casos que a taxonomia cobre bem.

### Próximos passos para calibração

1. Anotar manualmente ~50 pares (skill_raw → taxonomy) para ter ground truth real.
2. Recalcular a curva sobre o ground truth anotado.
3. Considerar **thresholds distintos por categoria** (skills técnicas têm distribuição de scores diferente de skills comportamentais).
4. Para scores em [0.55, 0.70], considerar reranking com LLM para casos ambíguos.